# Chapter 15 — Exercise Solutions## RLHF with TRL[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

### Solution 15.1: Training Monitor

In [ ]:
import numpy as npfrom typing import List, Dictdef training_monitor(metrics: Dict[str, List[float]], thresholds: dict,                     window: int = 20) -> List[dict]:    """    Automated training monitor for RLHF/PPO training.    Returns list of alerts with severity and recommended action.    """    alerts = []    def smooth(values, w=window):        if len(values) < w: return values        return np.convolve(values, np.ones(w)/w, mode='valid').tolist()    # 1. Reward collapse: reward growing too fast (>2 std from recent trend)    if 'reward' in metrics:        r = metrics['reward']        if len(r) > 50:            recent_growth = np.mean(np.diff(r[-30:]))            baseline_growth = np.mean(np.diff(r[10:40]))            if recent_growth > baseline_growth * 3 and recent_growth > 0.05:                alerts.append({                    'type': 'REWARD_COLLAPSE',                    'severity': 'HIGH',                    'message': f'Reward growing {recent_growth/max(baseline_growth,1e-9):.1f}x faster than baseline',                    'action': 'Sample 50 high-scoring outputs for human review; increase KL penalty β',                    'step': len(r)                })    # 2. Mode collapse: entropy dropping below threshold    if 'entropy' in metrics:        ent = metrics['entropy']        min_entropy = thresholds.get('min_entropy', 1.5)        if len(ent) > 10 and np.mean(ent[-10:]) < min_entropy:            alerts.append({                'type': 'MODE_COLLAPSE',                'severity': 'HIGH',                'message': f'Entropy {np.mean(ent[-10:]):.3f} < threshold {min_entropy}',                'action': 'Increase entropy coefficient c_2; check for degenerate policy modes',                'step': len(ent)            })    # 3. Value function divergence    if 'vf_loss' in metrics and 'policy_loss' in metrics:        vf = metrics['vf_loss']; pl = metrics['policy_loss']        if len(vf) > 10:            ratio = np.mean(vf[-10:]) / max(np.mean(pl[-10:]), 1e-9)            if ratio > thresholds.get('max_vf_ratio', 10):                alerts.append({                    'type': 'VALUE_DIVERGENCE',                    'severity': 'MEDIUM',                    'message': f'VF loss / Policy loss ratio = {ratio:.1f} (threshold: {thresholds.get("max_vf_ratio",10)})',                    'action': 'Reduce value function learning rate; check value function architecture',                    'step': len(vf)                })    # 4. KL divergence out of target range    if 'kl' in metrics:        kl = metrics['kl']        kl_target = thresholds.get('kl_target', 6.0)        if len(kl) > 10:            recent_kl = np.mean(kl[-10:])            if recent_kl > kl_target * 2:                alerts.append({                    'type': 'KL_EXPLOSION',                    'severity': 'HIGH',                    'message': f'KL {recent_kl:.2f} >> target {kl_target} (ratio={recent_kl/kl_target:.1f}x)',                    'action': 'Increase β (KL penalty); reduce learning rate; check for reward hacking',                    'step': len(kl)                })            elif recent_kl < kl_target * 0.1:                alerts.append({                    'type': 'KL_COLLAPSE',                    'severity': 'LOW',                    'message': f'KL {recent_kl:.4f} << target {kl_target} (policy not moving)',                    'action': 'Decrease β; check if gradients are flowing properly',                    'step': len(kl)                })    return alerts if alerts else [{'type': 'OK', 'severity': 'NONE', 'message': 'All metrics within bounds'}]# Test with simulated metricsnp.random.seed(42)steps = 200# Scenario 1: Reward collapse at step 150metrics_collapse = {    'reward': list(np.log1p(np.arange(200)/20) + np.random.normal(0,0.1,200)),    'kl':     list(np.ones(200)*5.5 + np.random.normal(0,0.5,200)),    'entropy': list(np.linspace(4.0, 1.2, 200) + np.random.normal(0,0.1,200)),    'policy_loss': list(np.random.exponential(0.5, 200)),    'vf_loss': list(np.random.exponential(2.0, 200)),}# Inject reward spike at step 150metrics_collapse['reward'][150:] = [v*3 for v in metrics_collapse['reward'][150:]]thresholds = {'min_entropy': 1.5, 'kl_target': 6.0, 'max_vf_ratio': 8.0}alerts = training_monitor(metrics_collapse, thresholds)print("Training Monitor Alerts:")for alert in alerts:    print(f"  [{alert['severity']}] {alert['type']}: {alert['message']}")    if 'action' in alert: print(f"    Action: {alert['action']}")# YOUR TURN: Add a fifth alert: "plateau" — reward hasn't improved for N steps# Threshold: reward moving average hasn't increased by >0.01 in last 50 steps

### Solution 15.2: KL and β Analysis

In [ ]:
import numpy as np, matplotlib.pyplot as pltdef simulate_rlhf_training(beta, n_steps=300, lr=1e-4):    """    Simulates RLHF training dynamics for a given β value.    Returns: reward_scores, kl_divergences, human_quality_scores    """    np.random.seed(42)    reward = [0.0]    kl     = [0.0]    human_quality = [0.0]  # true quality (not reward model score)    # Simulate: reward and KL increase together; human quality plateaus earlier    for step in range(1, n_steps):        # Reward: increases with learning but can overshoot (hacking)        noise = np.random.normal(0, 0.05)        if beta < 0.1:  # low beta: aggressive optimisation → hacking after ~150 steps            r = min(2.5, np.log1p(step/30) + noise + max(0, (step-150)/100))        else:           # high beta: conservative → reward grows steadily            r = min(1.5, np.log1p(step/50) * beta/0.1 + noise)        reward.append(r)        # KL grows with reward but is controlled by beta        kl_new = kl[-1] + lr * r / beta + np.random.normal(0, 0.1)        kl.append(max(0, kl_new))        # Human quality: plateaus before reward model score (overoptimisation)        plateau_step = 200 if beta >= 0.1 else 100        hq = min(1.0, np.log1p(step/plateau_step) + np.random.normal(0, 0.05))        human_quality.append(hq)    return reward, kl, human_qualitybetas = [0.05, 0.1, 0.3, 0.5]fig, axes = plt.subplots(1, 3, figsize=(16, 4))colors = ['tomato', 'steelblue', 'seagreen', 'purple']for (beta, color) in zip(betas, colors):    r, k, hq = simulate_rlhf_training(beta)    axes[0].plot(r,  label=f'β={beta}', color=color, lw=1.5)    axes[1].plot(k,  label=f'β={beta}', color=color, lw=1.5)    axes[2].plot(hq, label=f'β={beta}', color=color, lw=1.5)for ax, title in zip(axes, ['Reward Model Score', 'KL Divergence', 'Human Quality']):    ax.set_title(title); ax.set_xlabel('Step')    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)plt.suptitle('RLHF Training: Effect of KL Penalty β', fontsize=12)plt.tight_layout(); plt.show()print("Observations:")print("β=0.05: high reward, high KL → likely reward hacking, human quality diverges from RM score")print("β=0.10: balanced: good reward with controlled KL → optimal tradeoff in most cases")print("β=0.30: conservative: lower reward but very stable, human quality closely tracks RM score")print("β=0.50: too conservative: model barely moves from reference policy")print()print("Rule of thumb: start at β=0.1, monitor KL, adjust adaptively")print("If KL > 2× target: increase β; if reward plateau too early: decrease β")# YOUR TURN: Find the β that maximises human_quality[-1] across the simulations# Is it the same β that maximises reward[-1]? If different, what does this tell you?